# 8 — Post-Holdout Frozen-Snapshot Validation

## Objective

This notebook performs a **strictly post-holdout validation** of the frozen Missed Delivery model.

It reconstructs what would have been known at a historical scoring date after the original analytical dataset ended, builds the same 154 D-7 features, scores the frozen model, and only then joins the realised Missed Delivery outcome.

This notebook is **not** a production pipeline and does not automate daily scoring.

Scientific rule:

```text
snapshot / scoring date = T
target date             = T + 7 calendar days

features                = information available at T or earlier
prediction              = frozen model
actual outcome          = joined only after prediction
```

The original analytical dataset ended on 2026-08-25. Therefore this notebook must use `scoring_date > 2026-08-25`.


In [ ]:
import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants

from pyspark.sql import functions as F
from pyspark.sql import types as T

from datetime import date, timedelta
from functools import reduce
from operator import or_

import json
import pandas as pd
import numpy as np
import mlflow


## 1. Load historical Fabric sources

Unlike the daily-scoring notebook, this validation uses the historical frozen-order snapshot table.

`Fact_OTD` is loaded for the realised label only. It must not be used to create predictors.

In [ ]:
orders_hist = spark.read.synapsesql(
    "Logistica_DW.gld_erp.FACT_ENVIOS_PREV_HIST"
)

materials = spark.read.synapsesql(
    "Logistica_DW.gld_erp.dim_material"
)

fact_otd = spark.read.synapsesql(
    "WH_P3-2.dbo.Fact_OTD"
)

operational_raw = spark.read.synapsesql(
    "BRZ_SLV_Lakegistica.erp.fact_dim_production_rates_followup"
)

print("Historical validation sources loaded")

## 2. Select one frozen post-holdout snapshot

Start with one fully matured validation date.

Example:

```text
scoring_date = 2026-08-26
target_date  = 2026-09-02
```

Do not change the frozen model, calibrator, feature contract, or threshold after observing the result.


In [ ]:
# Última fecha que ya tiene predicciones guardadas
predictions_hist = spark.table("md_predictions")

SCORING_DATE = (
    predictions_hist
    .agg(F.max("scoring_date").alias("last_scoring_date"))
    .first()["last_scoring_date"]
) + timedelta(days=1)

# El modelo predice exactamente D+7
TARGET_DATE = SCORING_DATE +  timedelta(days=7)

SCORING_DATE_KEY = int(SCORING_DATE.strftime("%Y%m%d"))
TARGET_DATE_KEY = int(TARGET_DATE.strftime("%Y%m%d"))

print("Scoring date:", SCORING_DATE)
print("Target date:", TARGET_DATE)
print("Snapshot DateKeyCreated:", SCORING_DATE_KEY)
print("Delivery DateKey:", TARGET_DATE_KEY)

## 3. Reconstruct the frozen order universe

The historical snapshot must contain the orders that were pending at `SCORING_DATE` for delivery exactly seven calendar days later.

`request` is retained as metadata and is **not** one of the 154 model predictors.

In [ ]:
pedidos = (
    orders_hist.alias("f")
    .join(
        materials.alias("d"),
        F.col("f.Materialkey") == F.col("d.MaterialKey"),
        "inner"
    )
    .filter(
        (F.col("f.DateKeyCreated") == F.lit(SCORING_DATE_KEY)) &
        (F.col("f.DateKey") == F.lit(TARGET_DATE_KEY)) &
        (F.col("d.Centro") == F.lit("ES00")) &
        (F.col("d.GrupoEstratgPlanif") == F.lit("ZA"))
    )
    .groupBy(
        F.col("d.Grupo_raiz").alias("Grupo_raiz")
    )
    .agg(
        F.sum("f.ctd_pendiente").alias("request")
    )
    .withColumn("scoring_date", F.lit(SCORING_DATE).cast("date"))
    .withColumn("target_date", F.lit(TARGET_DATE).cast("date"))
    .select(
        "Grupo_raiz",
        "scoring_date",
        "target_date",
        "request"
    )
)

pedidos_count = pedidos.count()
print("Frozen snapshots groups: ", pedidos_count)

assert pedidos_count > 0, (
    f"No frozen obligations found for"
    f"{SCORING_DATE} --> {TARGET_DATE}"
)
display(pedidos)

## 4. Point-in-time categorical context — REQUIRED

The frozen model requires:

- `Grupo_raiz`
- `customer`
- `uat`
- `destination`

For this prospective/post-holdout validation, `customer`, `uat`, and `destination` must come from the frozen order snapshot or another point-in-time source available no later than `SCORING_DATE`.

**Do not obtain these predictors from realised `Fact_OTD` rows.**

The next cell lists candidate columns from `FACT_ENVIOS_PREV_HIST`. Set the three mappings only after verifying the physical meaning of the columns.


In [ ]:
candidate_context_columns = [
    c for c in orders_hist.columns
    if any(
        token in c.lower()
        for token in [
            "client", "customer", "uat",
            "dest", "direc", "ship", "address"
        ]
    )
]

print("Candidate point-in-time context columns:")
for c in candidate_context_columns:
    print(" -", c)


## 3. Business Context

The registered model also requires categorical context:

- planning group,
- customer,
- UAT,
- destination.

For the controlled MVP, these attributes are derived from the historical delivery context associated with the planning group.

In [ ]:
print("ORDER COLUMNS")
for c in orders_hist.columns:
    if any(
        x in c.lower()
        for x in [
            "client",
            "customer",
            "uat",
            "dest",
            "direc",
            "date",
            "fecha",
        ]
    ):
        print(c)

print("\nFACT_OTD COLUMNS")
for c in fact_otd.columns:
    if any(
        x in c.lower()
        for x in [
            "client",
            "customer",
            "uat",
            "dest",
            "direc",
            "date",
            "fecha",
        ]
    ):
        print(c)

## 4. Business Context

The registered model also requires categorical context:

- planning group,
- customer,
- UAT,
- destination.

For the controlled MVP, these attributes are derived from the historical delivery context associated with the planning group.

In [ ]:
contexto = (
    fact_otd.alias("fact")
    .join(
        materials.alias("d"),
        (F.col("d.Material") == F.col("fact.Id_item")) &
        (F.col("d.Centro") == F.lit("ES00")) &
        (F.col("d.GrupoEstratgPlanif") == F.lit("ZA")),
        "inner"
    )
    .groupBy(
        F.col("d.Grupo_raiz").alias("Grupo_raiz")
    )
    .agg(
        F.max("fact.Id_ClienteBaan").alias("customer"),
        F.max("fact.Id_UAT").alias("uat"),
        F.max("fact.Direccion_Entrega").alias("destination")
    )
)

In [ ]:
scoring_universe = (
    pedidos
    .join(
        contexto,
        on="Grupo_raiz",
        how="inner"
    )
)

display(scoring_universe)

## 4. Operational History

The model uses operational information observed before the target delivery date.

The required exact calendar lags are:

- D-7
- D-14
- D-21
- D-28

A 35-day extraction window is used to provide sufficient recent operational history while keeping the MVP dataset compact.

In [ ]:
operational_history_full = (
    operational_raw.alias("f")
    .join(
        materials.alias("d"),
        (F.col("f.Referencia") == F.col("d.Material")) &
        (F.col("d.Centro") == F.lit("ES00")),
        "inner"
    )
    .filter(
        F.col("f.Werks") == "ES00"
    )
    .withColumn(
        "Fecha",
        F.to_date("f.DateRegister")
    )
    .select(
        "Fecha",
        F.col("d.Grupo_raiz").alias("Grupo_raiz"),

        "f.FabricacionSemana",
        "f.RitmoSemana",
        "f.CumpFabRit",

        "f.ExpedidasSemana",
        "f.Demanda",
        "f.CumpExpDem",

        "f.StockFin",
        "f.StockReal",
        "f.CumpStock",

        "f.AcumFab",
        "f.AcumRitmo",
        "f.PorcenCump",

        "f.AcumEnvio",
        "f.AcumDemanda",
        "f.PorcenCumpto",

        "f.EntregasPrev",
        "f.DemandActual",
        "f.CumpTotal"
    )
)

In [ ]:
operational_history = (
    operational_history_full
    .filter(
        (F.col("Fecha") >= F.date_sub(F.lit(TARGET_DATE), 28)) &
        (F.col("Fecha") <= F.date_sub(F.lit(TARGET_DATE), 7))
    )
)

In [ ]:
display(operational_history)

## 5. Exact Temporal Lag Dates

For each scoring observation, the operational dates required by the model are derived directly from the target date.

Missing exact dates are preserved as unavailable rather than replaced by nearest observations.

In [ ]:
scoring_base = (
    scoring_universe
    .withColumn(
        "date_D7",
        F.date_sub("target_date", 7)
    )
    .withColumn(
        "date_D14",
        F.date_sub("target_date", 14)
    )
    .withColumn(
        "date_D21",
        F.date_sub("target_date", 21)
    )
    .withColumn(
        "date_D28",
        F.date_sub("target_date", 28)
    )
)

display(
    scoring_base.select(
        "Grupo_raiz",
        "target_date",
        "date_D7",
        "date_D14",
        "date_D21",
        "date_D28"
    )
)

In [ ]:
operational_cols = [
    "FabricacionSemana",
    "RitmoSemana",
    "CumpFabRit",
    "ExpedidasSemana",
    "Demanda",
    "CumpExpDem",
    "StockFin",
    "StockReal",
    "CumpStock",
    "AcumFab",
    "AcumRitmo",
    "PorcenCump",
    "AcumEnvio",
    "AcumDemanda",
    "PorcenCumpto",
    "EntregasPrev",
    "DemandActual",
    "CumpTotal"
]

In [ ]:
duplicate_history = (
    operational_history
    .groupBy(
        "Fecha",
        "Grupo_raiz"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

duplicate_count = duplicate_history.count()

print(
    "Duplicate Fecha + Grupo_raiz keys:",
    duplicate_count
)

assert duplicate_count == 0, (
    "Operational history contains duplicate "
    f"Fecha + Grupo_raiz keys: {duplicate_count}"
)

In [ ]:
hist_D7 = operational_history.select(
    F.col("Grupo_raiz"),
    F.col("Fecha").alias("date_D7"),
    *[
        F.col(c).alias(f"{c}_D7")
        for c in operational_cols
    ]
)

features_df = (
    scoring_base
    .join(
        hist_D7,
        on=["Grupo_raiz", "date_D7"],
        how="left"
    )
)

In [ ]:
before_count = scoring_base.count()
after_count = features_df.count()

print("Rows before D7 join:", before_count)
print("Rows after D7 join:", after_count)

assert before_count == after_count, (
    f"D7 join changed row count: "
    f"{before_count} -> {after_count}"
)

In [ ]:
from functools import reduce
from operator import or_

lag_has_any_value = reduce(
    or_,
    [
        F.col(f"{column}_D7").isNotNull()
        for column in operational_cols
    ],
)

features_df = features_df.withColumn(
    "history_available_D7",
    F.when(
        lag_has_any_value,
        F.lit(1.0)
    )
    .otherwise(F.lit(0.0))
)

In [ ]:
from functools import reduce
from operator import or_

def add_exact_lag(df, history_df, lag):

    date_col = f"date_D{lag}"

    hist = history_df.select(
        F.col("Grupo_raiz"),
        F.col("Fecha").alias(date_col),
        *[
            F.col(c).alias(f"{c}_D{lag}")
            for c in operational_cols
        ]
    )

    before_count = df.count()

    df = df.join(
        hist,
        on=["Grupo_raiz", date_col],
        how="left"
    )

    after_count = df.count()

    assert before_count == after_count, (
        f"D{lag} join changed row count: "
        f"{before_count} -> {after_count}"
    )

    lag_has_any_value = reduce(
        or_,
        [
            F.col(f"{column}_D{lag}").isNotNull()
            for column in operational_cols
        ]
    )

    df = df.withColumn(
        f"history_available_D{lag}",
        F.when(
            lag_has_any_value,
            F.lit(1.0)
        ).otherwise(F.lit(0.0))
    )

    print(
        f"D{lag} join rows:",
        before_count,
        "->",
        after_count
    )

    return df

In [ ]:
features_df = scoring_base

for lag in [7, 14, 21, 28]:

    features_df = add_exact_lag(
        features_df,
        operational_history,
        lag
    )

In [ ]:
features_df = features_df.withColumn(
    "available_lag_count",
    F.col("history_available_D7") +
    F.col("history_available_D14") +
    F.col("history_available_D21") +
    F.col("history_available_D28")
)

In [ ]:
operational_history_dates = (
    operational_history_full
    .select(
        "Fecha",
        "Grupo_raiz"
    )
    .dropDuplicates()
)

history_support = (
    scoring_universe.alias("s")
    .join(
        operational_history_dates.alias("h"),
        (F.col("s.Grupo_raiz") == F.col("h.Grupo_raiz")) &
        (F.col("h.Fecha") < F.col("s.target_date")),
        "left"
    )
    .groupBy(
        F.col("s.Grupo_raiz"),
        F.col("s.scoring_date"),
        F.col("s.target_date")
    )
    .agg(
        F.count("h.Fecha")
        .cast("double")
        .alias("history_records_before_D")
    )
)

before_count = features_df.count()

features_df = (
    features_df
    .join(
        history_support,
        on=[
            "Grupo_raiz",
            "scoring_date",
            "target_date"
        ],
        how="left"
    )
)

after_count = features_df.count()

print(
    "Rows before history support join:",
    before_count
)

print(
    "Rows after history support join:",
    after_count
)

assert before_count == after_count, (
    "history_support join changed row count: "
    f"{before_count} -> {after_count}"
)

## 6. Operational Gap Features

Operational gap features quantify the difference between production, shipment, stock and planning quantities at each historical lag.

In [ ]:
for lag in [7, 14, 21, 28]:

    features_df = (
        features_df

        .withColumn(
            f"production_gap_D{lag}",
            F.col(f"FabricacionSemana_D{lag}") -
            F.col(f"RitmoSemana_D{lag}")
        )

        .withColumn(
            f"shipment_demand_gap_D{lag}",
            F.col(f"ExpedidasSemana_D{lag}") -
            F.col(f"Demanda_D{lag}")
        )

        .withColumn(
            f"stock_gap_D{lag}",
            F.col(f"StockReal_D{lag}") -
            F.col(f"StockFin_D{lag}")
        )

        .withColumn(
            f"accumulated_production_gap_D{lag}",
            F.col(f"AcumFab_D{lag}") -
            F.col(f"AcumRitmo_D{lag}")
        )

        .withColumn(
            f"accumulated_delivery_gap_D{lag}",
            F.col(f"AcumEnvio_D{lag}") -
            F.col(f"AcumDemanda_D{lag}")
        )

        .withColumn(
            f"planned_delivery_gap_D{lag}",
            F.col(f"EntregasPrev_D{lag}") -
            F.col(f"DemandActual_D{lag}")
        )
    )

In [ ]:
delta_bases = [
    "CumpStock",
    "stock_gap",
    "CumpExpDem",
    "shipment_demand_gap",
    "CumpFabRit",
    "production_gap",
    "PorcenCumpto",
    "accumulated_delivery_gap",
    "PorcenCump",
    "accumulated_production_gap",
    "CumpTotal",
    "planned_delivery_gap",
    "Demanda",
    "DemandActual",
    "EntregasPrev"
]

In [ ]:
for base in delta_bases:

    features_df = (
        features_df

        .withColumn(
            f"{base}_delta_D7_D14",
            F.col(f"{base}_D7") -
            F.col(f"{base}_D14")
        )

        .withColumn(
            f"{base}_delta_D14_D28",
            F.col(f"{base}_D14") -
            F.col(f"{base}_D28")
        )

        .withColumn(
            f"{base}_delta_D7_D28",
            F.col(f"{base}_D7") -
            F.col(f"{base}_D28")
        )
    )

In [ ]:
features_df = (
    features_df
    .withColumn(
        "month",
        F.month("target_date").cast("double")
    )
    .withColumn(
        "day_of_week",
        ((F.dayofweek("target_date") + 5) % 7).cast("double")
    )
    .withColumn(
        "week_of_year",
        F.weekofyear("target_date").cast("double")
    )
)

In [ ]:
display(
    features_df.select(
        "target_date",
        "month",
        "day_of_week",
        "week_of_year"
    )
)

In [ ]:
import json

with open(
    "./builtin/feature_schema.json",
    "r",
    encoding="utf-8"
) as f:
    feature_schema = json.load(f)

expected_features = feature_schema["expected_features"]
categorical_features = feature_schema["categorical_features"]
numerical_features = feature_schema["numerical_features"]

print("Expected features:", len(expected_features))
print("Categorical:", len(categorical_features))
print("Numerical:", len(numerical_features))

In [ ]:
missing_features = [
    c for c in expected_features
    if c not in features_df.columns
]

extra_features = [
    c for c in features_df.columns
    if c not in expected_features
]

print("Features currently available:", len(features_df.columns))
print("Missing model features:", len(missing_features))
print(missing_features)

print("\nNon-model / metadata columns:", len(extra_features))
print(extra_features)

In [ ]:
integer_features = [
    "history_available_D7",
    "history_available_D14",
    "history_available_D21",
    "history_available_D28",
    "available_lag_count",
    "history_records_before_D",
    "month",
    "day_of_week",
    "week_of_year"
]

for c in integer_features:
    features_df = features_df.withColumn(
        c,
        F.col(c).cast("long")
    )

In [ ]:
for c in categorical_features:
    features_df = features_df.withColumn(
        c,
        F.col(c).cast("string")
    )

In [ ]:
float_features = [
    c for c in numerical_features
    if c not in integer_features
]

for c in float_features:
    features_df = features_df.withColumn(
        c,
        F.col(c).cast("double")
    )

In [ ]:
scoring_features = features_df.select(
    "scoring_date",
    "target_date",
    "request",
    *expected_features
)

print("Total output columns:", len(scoring_features.columns))
print("Model features:", len(expected_features))

In [ ]:
assert len(expected_features) == 154

missing_features = [
    c for c in expected_features
    if c not in scoring_features.columns
]

assert not missing_features, missing_features

assert scoring_features.columns[3:] == expected_features

final_count = scoring_features.count()

print("Final rows:", final_count)
print("Final columns:", len(scoring_features.columns))
print("Feature order check: PASS")

In [ ]:
(
    scoring_features
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("md_scoring_features")
)

In [ ]:
check_df = spark.table("md_scoring_features")

print("Rows:", check_df.count())
print("Columns:", len(check_df.columns))

display(check_df)

In [ ]:
check_df.groupBy(
    "history_available_D7",
    "history_available_D14",
    "history_available_D21",
    "history_available_D28"
).count().orderBy(
    F.desc("count")
).show(truncate=False)

In [ ]:
check_df.select(
    F.min("history_records_before_D").alias("min_history"),
    F.max("history_records_before_D").alias("max_history"),
    F.avg("history_records_before_D").alias("avg_history")
).show()

In [ ]:
display(
    check_df.select(
        "Grupo_raiz",
        "history_available_D7",
        "history_available_D14",
        "history_available_D21",
        "history_available_D28",
        "available_lag_count",
        "history_records_before_D",
        "StockReal_D7",
        "StockReal_D14",
        "StockReal_D21",
        "StockReal_D28"
    )
)